# Lab 08: Iceberg tables on HDFS

Iceberg is a table format. It keeps a small tree of metadata files next to the Parquet data files, which gives you UPDATE, DELETE, MERGE, snapshots and time travel on plain HDFS.

Run the cells from top to bottom. The notebook starts from a clean table each time.

In [ ]:
from pyspark.sql import SparkSession

WAREHOUSE = "hdfs://namenode:8020/user/lab/08/warehouse"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("lab-08-iceberg")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", WAREHOUSE)
    .config("spark.hadoop.dfs.replication", "1")
    .getOrCreate()
)
print("Spark", spark.version, "| catalog lab ->", WAREHOUSE)

def hdfs_ls(path):
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    return sorted(
        (s.getPath().getName(), s.getLen()) for s in fs.listStatus(p)
    )

## 1. Create a table

`lab` is the catalog name from the config above. `shop` is a namespace (like a database). The table lives under `warehouse/shop/orders`.

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS lab.shop")
spark.sql("DROP TABLE IF EXISTS lab.shop.orders PURGE")
spark.sql('''
    CREATE TABLE lab.shop.orders (
        order_id BIGINT,
        customer STRING,
        item STRING,
        qty INT,
        price DECIMAL(8, 2),
        status STRING
    ) USING iceberg
''')
spark.sql("DESCRIBE TABLE lab.shop.orders").show(truncate=False)

## 2. Insert (snapshot 1)

In [ ]:
spark.sql('''
    INSERT INTO lab.shop.orders VALUES
        (1, 'asha',  'notebook',   2, 3.50,  'new'),
        (2, 'bikash','pen',       10, 1.20,  'new'),
        (3, 'chen',  'backpack',   1, 24.00, 'new'),
        (4, 'dina',  'lamp',       1, 18.50, 'cancelled'),
        (5, 'eli',   'bottle',     3, 9.00,  'new')
''')
spark.sql("SELECT * FROM lab.shop.orders ORDER BY order_id").show()

## 3. Update and delete (snapshots 2 and 3)

HDFS files cannot be edited in place. Iceberg writes new files and a new snapshot that points at them.

In [ ]:
spark.sql("UPDATE lab.shop.orders SET status = 'shipped' WHERE order_id IN (1, 2)")
spark.sql("DELETE FROM lab.shop.orders WHERE status = 'cancelled'")
spark.sql("SELECT * FROM lab.shop.orders ORDER BY order_id").show()

## 4. Merge (snapshot 4)

MERGE is an upsert: update the rows that match, insert the rest.

In [ ]:
spark.sql('''
    CREATE OR REPLACE TEMP VIEW incoming AS
    SELECT * FROM VALUES
        (3, 'chen',  'backpack', 1, 22.00, 'shipped'),
        (6, 'farah', 'headphones', 1, 39.90, 'new')
    AS t(order_id, customer, item, qty, price, status)
''')
spark.sql('''
    MERGE INTO lab.shop.orders AS t
    USING incoming AS s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
''')
spark.sql("SELECT * FROM lab.shop.orders ORDER BY order_id").show()

## 5. Snapshots

Every change above is one row here. This is a metadata table, queried like any other.

In [ ]:
snaps = spark.sql('''
    SELECT snapshot_id, committed_at, operation,
           summary['added-records'] AS added, summary['deleted-records'] AS deleted
    FROM lab.shop.orders.snapshots
    ORDER BY committed_at
''')
snaps.show(truncate=False)

## 6. Time travel

Read the table as it was after the first snapshot, then as of the time of the second one.

In [ ]:
ids = [r.snapshot_id for r in snaps.collect()]
spark.sql(f"SELECT * FROM lab.shop.orders VERSION AS OF {ids[0]} ORDER BY order_id").show()

In [ ]:
second_time = spark.sql('''
    SELECT date_format(committed_at, 'yyyy-MM-dd HH:mm:ss.SSS')
    FROM lab.shop.orders.snapshots ORDER BY committed_at LIMIT 1 OFFSET 1
''').first()[0]
print("as of", second_time)
spark.sql(f"SELECT * FROM lab.shop.orders TIMESTAMP AS OF '{second_time}' ORDER BY order_id").show()

## 7. Schema evolution

Adding or renaming a column only changes metadata. No data file is rewritten, and old rows read as NULL in the new column.

In [ ]:
spark.sql("ALTER TABLE lab.shop.orders ADD COLUMN coupon STRING")
spark.sql("ALTER TABLE lab.shop.orders RENAME COLUMN qty TO quantity")
spark.sql("INSERT INTO lab.shop.orders VALUES (7, 'gita', 'lamp', 2, 18.50, 'new', 'SPRING10')")
spark.sql("SELECT * FROM lab.shop.orders ORDER BY order_id").show()

## 8. Look at the files in HDFS

`metadata/` holds `vN.metadata.json` (one per table change), manifest lists and manifests. `data/` holds the Parquet files. `version-hint.text` says which `vN` is current.

In [ ]:
table_dir = f"{WAREHOUSE}/shop/orders"
for folder in ["", "/metadata", "/data"]:
    print(table_dir + folder)
    for name, size in hdfs_ls(table_dir + folder):
        print(f"   {size:>8}  {name}")

The `files` metadata table lists the same data files from inside SQL:

In [ ]:
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM lab.shop.orders.files").show(truncate=False)

In [ ]:
spark.stop()